# Figure 4
Load pre-computed data and generate the figure.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from scipy import stats

from conus_biomass import dir_info
from conus_biomass.make_figures import maps

# Load saved figure data
save_dir = "figure_data/figure_4/"

biomass_delta_actual = xr.open_dataset(save_dir + "biomass_delta_actual.nc")["biomass_delta_actual"]
biomass_delta_modeled = xr.open_dataset(save_dir + "biomass_delta_modeled.nc")[
    "biomass_delta_modeled"
]
ds_binned = xr.open_dataset(save_dir + "ds_binned_actual.nc")
ds_binned_counts = xr.open_dataset(save_dir + "ds_binned_counts_actual.nc")
ds_binned_modeled = xr.open_dataset(save_dir + "ds_binned_modeled.nc")
ds_binned_modeled_counts = xr.open_dataset(save_dir + "ds_binned_counts_modeled.nc")

with open(save_dir + "target_crs.txt") as f:
    target_crs = f.read()

In [ ]:
fig, axs = plt.subplots(
    1, 4, figsize=(28, 9), gridspec_kw={"width_ratios": [0.05, 1, 1, 1]}, constrained_layout=True
)

font_settings = {
    "font.size": 28,
    "axes.titlesize": 30,
    "axes.labelsize": 28,
    "legend.fontsize": 28,
    "xtick.labelsize": 28,
    "ytick.labelsize": 28,
}
plt.rcParams.update(font_settings)
cmap = plt.cm.BrBG
clims = [-30, 30]
shp_native = maps.SHP_WESTERN.to_crs(target_crs)


def setup_ax(ax):
    ax.set_axis_off()
    shp_native.boundary.plot(ax=ax, color="gray", linewidth=1)


cbar_ax = axs[0]

ax = axs[1]
im = biomass_delta_actual.plot(ax=ax, vmin=clims[0], vmax=clims[1], cmap=cmap, add_colorbar=False)
ax.set_title("Mean Plot-level Observed ΔAGB")
setup_ax(ax)
# ax.text(0.02, 0.98, "a", transform=ax.transAxes, fontsize=28 * 1.25, fontweight="bold", va="top")

ax = axs[2]
im = biomass_delta_modeled.plot(ax=ax, vmin=clims[0], vmax=clims[1], cmap=cmap, add_colorbar=False)
ax.set_title("Mean Plot-level Modeled ΔAGB")
setup_ax(ax)

cbar = fig.colorbar(im, cax=cbar_ax, orientation="vertical", extend="both")
cbar.set_label("ΔAGB between \n measurements (MgC/ha)")
cbar.ax.tick_params(labelsize=font_settings["xtick.labelsize"])
cbar_ax.yaxis.set_ticks_position("left")
cbar_ax.yaxis.set_label_position("left")
# ax.text(0.02, 0.98, "b", transform=ax.transAxes, fontsize=28 * 1.25, fontweight="bold", va="top")

ax = axs[3]
x = ds_binned["biomass_delta_actual"].where(ds_binned_counts["n_plots"] >= 10).values.flatten()
y = (
    ds_binned_modeled["biomass_delta_modeled"]
    .where(ds_binned_modeled_counts["n_plots"] >= 10)
    .values.flatten()
)
nanfilter = (~np.isnan(x)) * (~np.isnan(y))

xmin = -75
xmax = 50
ax.plot(x[nanfilter], y[nanfilter], ".", markersize=15, alpha=0.5)
x_clean = x[nanfilter]
y_clean = y[nanfilter]

slope, intercept, r, p, se = stats.linregress(x_clean, y_clean)
r2 = r**2

x_fit = np.array([np.nanmin(x_clean), np.nanmax(y_clean)])
ax.plot(
    x_fit, slope * x_fit + intercept, "--", color="k", label=f"fit (y={slope:.2f}x+{intercept:.2f})"
)
ax.text(0.02, 0.85, f"y={slope:.2f}x + {intercept:.2f}", transform=ax.transAxes, fontsize=24)
ax.text(0.02, 0.78, f"R² = {r2:.2f}", transform=ax.transAxes, fontsize=24)
ax.set_xlabel("Mean plot-level observed \n ΔAGB (MgC/ha)")
ax.set_ylabel("Mean plot-level modeled \n ΔAGB (MgC/ha)")
ax.plot([xmin, xmax], [xmin, xmax], "k")
ax.axhline(y=0, linestyle=":", color="k")
ax.axvline(x=0, linestyle=":", color="k")
ax.set_xlim([xmin, xmax])
ax.set_ylim([xmin, xmax])
ticks = np.arange(xmin + 25, xmax + 1, 25)  # adjust step size as needed
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.tick_params(axis="y", which="both", left=True, right=True, direction="in")
ax.tick_params(axis="x", which="both", bottom=True, top=True, direction="in")

panel_labels = ["A", "B", "C"]
for i, ax in enumerate(axs[1:]):
    ax.text(
        0.02,
        0.98,
        panel_labels[i],
        transform=ax.transAxes,
        fontsize=28 * 1.25,
        fontweight="bold",
        va="top",
    )

fig.get_layout_engine().set(w_pad=0, h_pad=0, hspace=0, wspace=0.02)
fig.savefig(dir_info.dir_figures + "Figure4.jpg", bbox_inches="tight", dpi=300)